# Scenario 3: Multi-Agent Research System

**Domains tested:** Agentic Architecture (D1 -- 27%), Tool Design & MCP (D2 -- 18%), Context Management (D5 -- 15%)

---

## Scenario Context

You are the architect at **InsightLab**, a market research firm. Your team is building a multi-agent research system that produces comprehensive, cited reports from diverse sources. A coordinator agent delegates tasks to specialized subagents:

- **Search Agent** -- finds relevant articles, papers, and data from web search results
- **Analysis Agent** -- analyzes quantitative data, identifies trends and patterns
- **Synthesis Agent** -- combines findings into coherent narratives, resolves conflicting sources

**Production requirements:**
- The coordinator must decompose research questions into subtasks that ensure broad coverage (not just the first angle it thinks of)
- Each subagent receives explicit context about its role, the overall question, and what other agents are investigating
- Subagents can run in parallel when their tasks are independent
- Every claim in the final report must include provenance (which source, which agent found it)
- Conflicting sources must be flagged, not silently resolved
- Errors in one subagent must not crash the entire pipeline

---

In [ ]:
import anthropic
import json
import concurrent.futures
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

---

## Part 1: Exam-Format Questions

---

### Question 1 (D1)

Your coordinator agent receives the research question: "What is the impact of remote work on employee productivity?" It decomposes the task into a single subtask: "Search for studies on remote work productivity." The final report covers only survey-based studies and misses economic data, industry-specific analysis, and longitudinal trends.

What is the root cause?

- **A)** The coordinator's task decomposition is too narrow -- it generated a single search query instead of multiple subtasks covering different research angles
- **B)** The search agent's tool is returning incomplete results
- **C)** The synthesis agent is filtering out relevant results
- **D)** The model's context window cannot hold all the research data

In [ ]:
q1 = {
    "correct": "A",
    "explanation": (
        "Narrow task decomposition is the most common failure in multi-agent systems. The coordinator "
        "should decompose 'impact of remote work on productivity' into multiple angles: (1) survey "
        "studies, (2) economic/GDP data, (3) industry-specific impacts (tech vs manufacturing), "
        "(4) longitudinal trends before/after COVID, (5) manager vs employee perspectives. "
        "A single subtask produces narrow coverage regardless of how good the search agent is. "
        "This tests Domain 1: task decomposition quality determines the ceiling of multi-agent output."
    )
}
print(f"Correct: {q1['correct']}")
print(f"\nExplanation: {q1['explanation']}")

### Question 2 (D1 + D5)

Your search agent and analysis agent are given the same research question but no information about what the other agent is investigating. Both agents independently search for the same studies and produce overlapping results, wasting tokens and time.

What is the best fix?

- **A)** Run the agents sequentially so the second agent can see the first agent's results
- **B)** Use a shared database that agents check before searching
- **C)** Reduce the number of agents to avoid overlap
- **D)** Have the coordinator pass explicit context to each subagent: their specific subtask, the overall question, and what other agents are responsible for, so each agent scopes its work appropriately

In [ ]:
q2 = {
    "correct": "D",
    "explanation": (
        "Explicit context passing from coordinator to subagents is the architectural solution. "
        "Each subagent should know: (1) its specific scope ('you are searching for economic data "
        "only'), (2) the overall research question (for relevance), and (3) what other agents are "
        "handling ('the search agent is covering survey studies -- do not duplicate'). Option A "
        "eliminates parallelism. Option B adds infrastructure complexity. Option C reduces "
        "coverage. This tests Domain 1 (coordinator-subagent communication) and Domain 5 "
        "(explicit context passing to avoid wasted computation)."
    )
}
print(f"Correct: {q2['correct']}")
print(f"\nExplanation: {q2['explanation']}")

### Question 3 (D1)

The search agent fails (API timeout) while the analysis agent completes successfully. The coordinator crashes and produces no report. What is the correct error handling pattern?

- **A)** Retry the search agent indefinitely until it succeeds
- **B)** Catch the error, produce a partial report from the analysis agent's results, and clearly mark which sections are incomplete due to the search failure
- **C)** Cache all previous search results so failures can use stale data
- **D)** Use a faster model for the search agent to avoid timeouts

In [ ]:
q3 = {
    "correct": "B",
    "explanation": (
        "Graceful degradation is the correct pattern: produce the best possible output with "
        "available data and clearly indicate what's missing. Infinite retries (A) can block the "
        "entire pipeline. Stale data (C) may be acceptable as a fallback but doesn't address the "
        "core issue. Faster models (D) don't guarantee no timeouts. The key principle: one agent's "
        "failure should not crash the entire system. The coordinator must handle partial results "
        "and communicate data gaps. This tests Domain 1: error propagation in multi-agent systems."
    )
}
print(f"Correct: {q3['correct']}")
print(f"\nExplanation: {q3['explanation']}")

### Question 4 (D2 + D5)

Your synthesis agent produces a report claiming "Remote work increases productivity by 13%." However, one source says +13%, another says -5%, and a third says no significant change. The agent averaged the numbers without flagging the conflict.

What architectural change addresses this?

- **A)** Require each finding to include provenance metadata (source, date, methodology) and instruct the synthesis agent to flag conflicting findings rather than silently resolving them
- **B)** Use a more accurate model for the synthesis agent
- **C)** Remove the synthesis agent and let humans combine the findings
- **D)** Filter out studies with negative results to avoid conflicts

In [ ]:
q4 = {
    "correct": "A",
    "explanation": (
        "Provenance tracking and conflict detection are essential for research systems. Every finding "
        "should carry metadata: which source, what methodology, when published. The synthesis agent "
        "must be explicitly instructed to detect and surface conflicts rather than averaging them. "
        "Option B doesn't address the structural issue. Option C abandons automation. Option D "
        "introduces selection bias. This tests Domain 2 (structured data passing between tools) "
        "and Domain 5 (context design for downstream agents)."
    )
}
print(f"Correct: {q4['correct']}")
print(f"\nExplanation: {q4['explanation']}")

### Question 5 (D1 + D2)

You need to run the search agent and analysis agent in parallel to reduce latency. Both agents use the same Claude API. What is the correct implementation pattern?

- **A)** Use Python `threading` to call the API simultaneously, since the API calls are I/O-bound
- **B)** Send both requests in a single API call with multiple system prompts
- **C)** Use the Batch API to submit both requests at once
- **D)** Use `asyncio` with `asyncio.gather()` to run both agent calls concurrently

In [ ]:
q5 = {
    "correct": "D",
    "explanation": (
        "asyncio.gather() is the idiomatic Python pattern for concurrent I/O-bound operations. "
        "Both agents make independent API calls that can run in parallel. Option A (threading / "
        "ThreadPoolExecutor) is also a valid and practical approach -- in fact, this notebook's "
        "hands-on build uses concurrent.futures.ThreadPoolExecutor, which works well for parallel "
        "API calls. asyncio is generally preferred for I/O-bound concurrency in new Python code, "
        "but ThreadPoolExecutor is a pragmatic alternative, especially when integrating with "
        "synchronous SDKs. Option B is not how the API works -- each request has one system prompt. "
        "Option C (Batch API) is for non-real-time processing and has higher latency (up to 24h). "
        "This tests Domain 1: knowing when and how to parallelize subagent execution."
    )
}
print(f"Correct: {q5['correct']}")
print(f"\nExplanation: {q5['explanation']}")

### Question 6 (D1 + D5)

Your coordinator agent sends the full research question plus all previous agent outputs as context to each subsequent subagent. As the research progresses, later agents receive 50,000+ tokens of context. Their output quality degrades.

What is the best mitigation?

- **A)** Use a model with a larger context window
- **B)** Limit each agent to 1,000 tokens of output to keep context small
- **C)** Have the coordinator summarize previous agent outputs into a concise briefing before passing to the next agent, preserving key findings and source references while reducing token count
- **D)** Skip the context passing entirely -- each agent works independently

In [ ]:
q6 = {
    "correct": "C",
    "explanation": (
        "Context quality matters more than context quantity. The coordinator should act as a context "
        "manager: summarize previous outputs, extract key findings and source references, and pass "
        "a focused briefing to the next agent. This preserves the information the agent needs while "
        "reducing noise. Option A doesn't fix attention degradation. Option B artificially limits "
        "useful output. Option D loses valuable cross-agent insights. This tests Domain 5: "
        "context compression and quality management in multi-agent pipelines."
    )
}
print(f"Correct: {q6['correct']}")
print(f"\nExplanation: {q6['explanation']}")

### Question 7 (D1 + D2 + D5)

You want the coordinator to decide dynamically whether to run additional research rounds based on the quality of initial findings. What is the best approach?

- **A)** Always run exactly 3 rounds of research regardless of results
- **B)** Let the user manually decide after each round
- **C)** Give the coordinator a `assess_coverage` tool that evaluates whether the current findings adequately answer the research question, and let it decide whether to dispatch additional subagent tasks
- **D)** Run as many rounds as possible until the context window is full

In [ ]:
q7 = {
    "correct": "C",
    "explanation": (
        "Dynamic task planning is a hallmark of well-designed agentic systems. The coordinator "
        "should have tools to assess its own progress and decide next steps. An assess_coverage "
        "tool could check: are all angles of the question covered? Are there contradictions that "
        "need additional sources? Is the confidence level sufficient? Option A is rigid and wasteful. "
        "Option B breaks automation. Option D has no quality signal. This tests all three domains: "
        "D1 (dynamic orchestration), D2 (tool design for self-assessment), D5 (managing growing context)."
    )
}
print(f"Correct: {q7['correct']}")
print(f"\nExplanation: {q7['explanation']}")

---

## Part 2: Hands-On Build

Build the multi-agent research system described in the scenario.

---

### Step 1: Define the subagent interfaces

In [ ]:
# Simulated search results (standing in for real web search)
SIMULATED_SOURCES = {
    "survey_studies": [
        {
            "title": "Stanford Remote Work Study 2023",
            "source": "Stanford University",
            "date": "2023-06",
            "finding": "Remote workers showed 13% productivity increase in structured environments",
            "methodology": "Randomized controlled trial, N=500"
        },
        {
            "title": "Microsoft Workplace Trends Report",
            "source": "Microsoft Research",
            "date": "2024-01",
            "finding": "Collaboration metrics declined 15% in fully remote teams compared to hybrid",
            "methodology": "Analysis of Teams usage data, N=60000"
        }
    ],
    "economic_data": [
        {
            "title": "Bureau of Labor Statistics Remote Work Analysis",
            "source": "BLS",
            "date": "2024-03",
            "finding": "Remote-capable industries grew GDP contribution by 4.2% from 2020-2024",
            "methodology": "National economic data analysis"
        }
    ],
    "industry_analysis": [
        {
            "title": "Tech Sector Remote Work Impact",
            "source": "Gartner",
            "date": "2023-11",
            "finding": "Software development velocity unchanged; design collaboration declined 20%",
            "methodology": "Survey of 200 tech companies"
        },
        {
            "title": "Manufacturing and Remote Oversight",
            "source": "McKinsey",
            "date": "2024-02",
            "finding": "Remote oversight reduced plant efficiency by 5-8% due to delayed decision-making",
            "methodology": "Case studies, 50 manufacturing facilities"
        }
    ]
}

print(f"Simulated {sum(len(v) for v in SIMULATED_SOURCES.values())} sources across {len(SIMULATED_SOURCES)} categories.")

In [ ]:
def run_search_agent(subtask: str, context: dict) -> dict:
    """Search agent: finds relevant sources for a specific research angle."""
    prompt = f"""You are a research search agent. Your task is to find relevant sources.

OVERALL RESEARCH QUESTION: {context['research_question']}
YOUR SPECIFIC SUBTASK: {subtask}
OTHER AGENTS ARE COVERING: {', '.join(context.get('other_agents_scope', []))}

Do NOT duplicate what other agents are searching for.

Available sources (simulated):
{json.dumps(SIMULATED_SOURCES, indent=2)}

Return a JSON object with:
- "findings": array of relevant sources (include title, source, date, finding, methodology)
- "coverage_notes": what angles this search covered
- "gaps": what angles remain unexplored

Return ONLY valid JSON."""

    try:
        response = client.messages.create(
            model=MODEL, max_tokens=2048,
            messages=[{"role": "user", "content": prompt}]
        )
        raw = response.content[0].text.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return {"status": "success", "agent": "search", "data": json.loads(raw)}
    except Exception as e:
        return {"status": "error", "agent": "search", "error": str(e)}


def run_analysis_agent(subtask: str, context: dict, findings: list) -> dict:
    """Analysis agent: identifies trends and conflicts in findings."""
    prompt = f"""You are a research analysis agent. Analyze the following findings.

OVERALL RESEARCH QUESTION: {context['research_question']}
YOUR SPECIFIC SUBTASK: {subtask}

FINDINGS TO ANALYZE:
{json.dumps(findings, indent=2)}

Return a JSON object with:
- "trends": array of identified trends (each with description, supporting_sources, confidence)
- "conflicts": array of conflicting findings (each with description, sources_in_conflict, suggested_resolution)
- "data_quality_notes": assessment of methodology quality across sources

IMPORTANT: Do NOT silently resolve conflicts. Flag them explicitly.

Return ONLY valid JSON."""

    try:
        response = client.messages.create(
            model=MODEL, max_tokens=2048,
            messages=[{"role": "user", "content": prompt}]
        )
        raw = response.content[0].text.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return {"status": "success", "agent": "analysis", "data": json.loads(raw)}
    except Exception as e:
        return {"status": "error", "agent": "analysis", "error": str(e)}


def run_synthesis_agent(context: dict, search_results: dict, analysis_results: dict) -> dict:
    """Synthesis agent: combines findings into a coherent report with provenance."""
    prompt = f"""You are a research synthesis agent. Produce a final report.

RESEARCH QUESTION: {context['research_question']}

SEARCH FINDINGS:
{json.dumps(search_results, indent=2)}

ANALYSIS:
{json.dumps(analysis_results, indent=2)}

RULES:
1. Every claim must cite its source (author/org + date)
2. Conflicting findings must be presented as conflicts, NOT averaged or silently resolved
3. Mark any sections with incomplete data due to agent failures
4. Include a confidence level (high/medium/low) for each major finding

Return a JSON object with:
- "title": report title
- "executive_summary": 2-3 sentence summary
- "sections": array of sections (each with heading, content, sources, confidence)
- "conflicts": array of unresolved conflicts
- "limitations": what this report does NOT cover

Return ONLY valid JSON."""

    try:
        response = client.messages.create(
            model=MODEL, max_tokens=4096,
            messages=[{"role": "user", "content": prompt}]
        )
        raw = response.content[0].text.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return {"status": "success", "agent": "synthesis", "data": json.loads(raw)}
    except Exception as e:
        return {"status": "error", "agent": "synthesis", "error": str(e)}


print("Subagent functions defined.")

### Step 2: Build the coordinator with task decomposition

In [ ]:
def decompose_research_question(question: str) -> list:
    """Use Claude to decompose a research question into subtasks."""
    prompt = f"""Decompose this research question into 3-5 specific, non-overlapping subtasks
for a research team. Each subtask should cover a different angle.

Research question: {question}

Return a JSON array of objects, each with:
- "subtask": specific research angle to investigate
- "agent": which agent should handle it ("search" or "analysis")
- "priority": 1-5 (1=highest)
- "depends_on": array of subtask indices this depends on (empty if independent)

Return ONLY valid JSON."""

    response = client.messages.create(
        model=MODEL, max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)


# Test decomposition
question = "What is the impact of remote work on employee productivity?"
subtasks = decompose_research_question(question)

print(f"Decomposed into {len(subtasks)} subtasks:\n")
for i, task in enumerate(subtasks):
    print(f"  {i+1}. [{task.get('agent', '?')}] {task.get('subtask', '?')}")
    print(f"     Priority: {task.get('priority', '?')}, Depends on: {task.get('depends_on', [])}")

### Step 3: Execute with parallel subagents and error handling

In [ ]:
def run_research_pipeline(question: str) -> dict:
    """Full research pipeline with parallel execution and error handling."""
    print(f"Research question: {question}\n")

    # Step 1: Decompose
    print("=== Step 1: Task Decomposition ===")
    subtasks = decompose_research_question(question)
    for i, t in enumerate(subtasks):
        print(f"  {i+1}. [{t.get('agent', '?')}] {t.get('subtask', '?')}")

    # Step 2: Run search agents in parallel
    print("\n=== Step 2: Parallel Search ===")
    search_tasks = [t for t in subtasks if t.get("agent") == "search"]
    all_scopes = [t.get("subtask", "") for t in search_tasks]

    search_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = {}
        for task in search_tasks:
            other_scopes = [s for s in all_scopes if s != task["subtask"]]
            context = {
                "research_question": question,
                "other_agents_scope": other_scopes
            }
            future = executor.submit(run_search_agent, task["subtask"], context)
            futures[future] = task["subtask"]

        for future in concurrent.futures.as_completed(futures):
            subtask_name = futures[future]
            result = future.result()
            status = result.get("status", "unknown")
            print(f"  [{status.upper()}] {subtask_name[:60]}")
            if status == "success":
                search_results.append(result["data"])
            else:
                print(f"    Error: {result.get('error', 'unknown')}")
                search_results.append({
                    "findings": [],
                    "coverage_notes": f"FAILED: {subtask_name}",
                    "gaps": [subtask_name]
                })

    # Step 3: Analysis
    print("\n=== Step 3: Analysis ===")
    all_findings = []
    for sr in search_results:
        all_findings.extend(sr.get("findings", []))

    analysis = run_analysis_agent(
        "Analyze all research findings for trends and conflicts",
        {"research_question": question},
        all_findings
    )
    print(f"  Analysis status: {analysis.get('status', 'unknown')}")

    # Step 4: Synthesis
    print("\n=== Step 4: Synthesis ===")
    report = run_synthesis_agent(
        {"research_question": question},
        search_results,
        analysis.get("data", {}) if analysis["status"] == "success" else {"error": "Analysis failed"}
    )
    print(f"  Synthesis status: {report.get('status', 'unknown')}")

    return report


# Run the full pipeline
result = run_research_pipeline("What is the impact of remote work on employee productivity?")

if result["status"] == "success":
    print("\n=== FINAL REPORT ===")
    report = result["data"]
    print(f"Title: {report.get('title', 'N/A')}")
    print(f"\nExecutive Summary: {report.get('executive_summary', 'N/A')}")
    print(f"\nSections: {len(report.get('sections', []))}")
    for s in report.get("sections", []):
        print(f"  - {s.get('heading', 'N/A')} (confidence: {s.get('confidence', 'N/A')})")
    if report.get("conflicts"):
        print(f"\nConflicts flagged: {len(report['conflicts'])}")
        for c in report["conflicts"]:
            print(f"  - {c}")

---

## Part 3: Failure Injections

---

### Failure 1: Narrow task decomposition

In [ ]:
print("=== FAILURE INJECTION: Single-angle decomposition ===")
print("What happens when we force a narrow decomposition?\n")

# Instead of decomposing, send a single subtask
narrow_context = {
    "research_question": "What is the impact of remote work on employee productivity?",
    "other_agents_scope": []
}

narrow_result = run_search_agent(
    "Search for studies on remote work productivity",  # single, broad query
    narrow_context
)

if narrow_result["status"] == "success":
    findings = narrow_result["data"].get("findings", [])
    gaps = narrow_result["data"].get("gaps", [])
    print(f"Findings: {len(findings)}")
    print(f"Gaps identified: {gaps}")

print("\n--- DIAGNOSIS ---")
print("A single broad query misses specific angles (economic data, industry differences,")
print("longitudinal trends). The coordinator's decomposition quality is the ceiling")
print("of the entire system's output quality.")
print("LESSON: Invest in task decomposition -- it determines coverage (Domain 1).")

### Failure 2: No provenance tracking

In [ ]:
print("=== FAILURE INJECTION: Missing provenance ===")
print("What happens when findings lack source attribution?\n")

# Strip provenance from findings
stripped_findings = [
    {"finding": "Remote work increases productivity by 13%"},
    {"finding": "Collaboration declines 15% in remote teams"},
    {"finding": "Remote oversight reduces efficiency by 5-8%"},
    {"finding": "Remote-capable industries grew GDP by 4.2%"}
]

# Ask synthesis agent to work with unattributed findings
prompt = f"""Synthesize these findings into a report about remote work productivity.

Findings:
{json.dumps(stripped_findings, indent=2)}

Write a 3-paragraph summary."""

response = client.messages.create(
    model=MODEL, max_tokens=1024,
    messages=[{"role": "user", "content": prompt}]
)

print(response.content[0].text)

print("\n--- DIAGNOSIS ---")
print("Without provenance, the report cannot cite sources. Claims like '13% productivity")
print("increase' are unverifiable. Worse, conflicting findings (13% increase vs 5-8%")
print("decrease) may be averaged instead of flagged.")
print("LESSON: Every finding must carry source, date, and methodology metadata (Domain 2).")

### Failure 3: Agent failure cascading

In [ ]:
print("=== FAILURE INJECTION: Cascading agent failure ===")
print("What happens when the search agent fails and there's no error handling?\n")

def run_search_agent_BROKEN(subtask, context):
    """Simulate a search agent that crashes."""
    raise ConnectionError("API timeout after 30 seconds")


# Without error handling: the crash propagates
print("Without error handling:")
try:
    result = run_search_agent_BROKEN("test", {})
    analysis = run_analysis_agent("test", {}, result["data"]["findings"])
    print("Pipeline completed")
except Exception as e:
    print(f"  CRASH: {type(e).__name__}: {e}")
    print("  The entire pipeline fails -- no report at all.")

# With error handling: graceful degradation
print("\nWith error handling:")
try:
    result = run_search_agent_BROKEN("test", {})
except Exception as e:
    result = {
        "status": "error",
        "agent": "search",
        "error": str(e),
        "data": {
            "findings": [],
            "coverage_notes": "SEARCH AGENT FAILED -- section will be incomplete",
            "gaps": ["all search results"]
        }
    }
    print(f"  Caught error: {e}")
    print(f"  Continuing with partial data: {result['data']['coverage_notes']}")
    print("  The report will be marked as incomplete but still produced.")

print("\n--- DIAGNOSIS ---")
print("Graceful degradation > total failure. Catch errors per-agent, produce partial")
print("reports, and clearly mark incomplete sections.")
print("LESSON: Error isolation in multi-agent systems is essential (Domain 1).")

---

## Key Takeaways

| Concept | Domain | Lesson |
|---------|--------|--------|
| Task decomposition quality | D1 | The coordinator's decomposition determines the ceiling of output quality |
| Explicit context passing | D1, D5 | Each subagent must know its scope, the overall question, and what others cover |
| Parallel execution | D1 | Use asyncio.gather() or ThreadPoolExecutor for independent subagent tasks |
| Provenance tracking | D2 | Every finding must carry source, date, and methodology metadata |
| Conflict detection | D2, D5 | Instruct agents to flag conflicts, never silently resolve them |
| Graceful degradation | D1 | One agent's failure should produce a partial report, not a crash |
| Context compression | D5 | Summarize previous agent outputs before passing to downstream agents |